In [3]:
import pandas as pd
import os
from features_reindex import get_feature, read_data, read_data_timecut
from sklearn.preprocessing import MinMaxScaler

In [4]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm'

time_spilt = True
test_bug = True

if test_bug:
    # feature_list = ['uniport_seq','uniport_esm']
    feature_list = ['uniport_seq']
    dga = 'disgenet'

    out_path = os.path.join(root,'results/temp')
    out_path_pred = out_path+'_pred/pred.pkl'
    time = 2019

merged_df = None

if time == 2017:
    time_feature_list = ['uniport_ppi_2017','ppi_2017_dw_80','uniport_exp','uniport_seq','uniport_esm']
elif time == 2019:
    time_feature_list = ['uniport_ppi_2019','ppi_2019_dw_40','uniport_bio','uniport_seq','uniport_esm','diffusion_2019']

for feature in time_feature_list:
    feature_df = get_feature(root, feature)
    if 'diffusion' in feature:
        pass
    else:
        feature_cols = [col for col in feature_df.columns if col.startswith('feature')]
        if feature_cols:
            scaler = MinMaxScaler()
            feature_df[feature_cols] = scaler.fit_transform(feature_df[feature_cols])

    # Rename columns starting with 'feature'
    feature_df.rename(columns={
        col: f"{feature}_{col}" if col.startswith('feature') else col
        for col in feature_df.columns
    }, inplace=True)

    # Merge iteratively to avoid keeping all DataFrames
    if merged_df is None:
        merged_df = feature_df
    else:
        merged_df = pd.merge(merged_df, feature_df, on='string_id', how='inner')
    del feature_df  # Free memory
name_list = feature_list + ['string_id']

merged_df = merged_df[[col for col in merged_df.columns if any(item in col for item in name_list)]]

if dga == 'disgenet':
    all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/dga_time_uniport.csv')
elif dga == 'opentarget':
    all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/ot_dga_time_uni.csv')
    all_df = all_df[all_df['score']>=0.4]

all_df = all_df[all_df['string_id'].isin(merged_df['string_id'])]
# all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/align_disgent_with_time.csv')

# methods = ['ooc','random_negative','pseudo_labeling','pseudo_labeling_mask']
# methods = ['random_negative','pseudo_labeling','pseudo_labeling_mask','pseudo_labeling_cluster_all_mask']
# methods = ['random_negative','random_negative_bagging','random_pos_negative_bagging']
methods = ['random_negative']
selected_diseases = ['ICD10_C43']

for disease in selected_diseases:
    print(disease,len(all_df[all_df['disease_id']==disease]))
    if time_spilt:
        df, y = read_data_timecut(disease, all_df, merged_df,time)
    else:
        df, y = read_data(disease, all_df, merged_df,time) 

ICD10_C43 47


In [5]:
test_idx = df[df['test']==1].index
train_idx = df[y==1].index.difference(test_idx)
df.drop(columns='test', inplace=True)

train_pos_df = df.loc[train_idx]
test_pos_df = df.loc[test_idx]
neg_num = 5*len(train_pos_df)
neg_df = df[y == 0]

In [9]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
from sklearn.metrics.pairwise import rbf_kernel

def compute_kernels(X_feature):
    gamma_kernel = dict()

    ratio_list = [2,4,8]

    nbrs = NearestNeighbors(n_neighbors=2).fit(X_feature)
    distances, _ = nbrs.kneighbors(X_feature)
    avg_nn_dist = np.mean(distances[:, 1])  # skip self-distance

    for ratio in ratio_list:
        gamma = 1 / (ratio * avg_nn_dist ** 2)
        K_full = rbf_kernel(X_feature, X_feature, gamma=gamma)
        K_full = 0.5 * (K_full + K_full.T)
        gamma_kernel[ratio] = K_full

    return gamma_kernel

In [10]:
gamma_kernel = compute_kernels(df.values)

## rbf + cv + test after 2019

## linear + L1 penalty + cv + test after 2019

## PCA features + rbf + cv + test after 2019

## time-split cv + test after 2019

In [ ]:
def owa_weights(model, m, alpha, tol=1e-12, max_iter=200):
    """
    Compute OWA ranking-place weights using one of four models.
    
    Parameters
    ----------
    model : int
        Which model to use:
            1 = Maximum Entropy (O'Hagan)
            2 = Extended Minimax Disparity (Amin & Emrouznejad)
    m : int
        Number of ranking places.
    alpha : float
        Target orness (0 <= alpha <= 1), typically >0.5 for ranking aggregation.
    tol : float, optional
        Tolerance for convergence (used in model 1).
    max_iter : int, optional
        Max iterations for convergence (used in model 1).
    
    Returns
    -------
    weights : list of float
        OWA weights of length m, summing to 1.
    """

    def calc_orness(w):
        idx = np.arange(1, m + 1)
        return (1.0 / (m - 1.0)) * np.sum((m - idx) * w)

    # ---------- Model 1: Maximum Entropy (geometric form) ----------
    def max_entropy():
        if abs(alpha - 0.5) < 1e-12:
            return np.ones(m) / m
        if abs(alpha - 1.0) < 1e-12:
            w = np.zeros(m); w[0] = 1.0; return w
        if abs(alpha - 0.0) < 1e-12:
            w = np.zeros(m); w[-1] = 1.0; return w

        def weights_from_r(r):
            exps = np.arange(m-1, -1, -1, dtype=float)
            v = r ** exps
            return v / v.sum()

        def orness_from_r(r):
            return calc_orness(weights_from_r(r))

        if alpha > 0.5:
            lo, hi = 1.0, 1e6
        else:
            lo, hi = 1e-6, 1.0

        for _ in range(max_iter):
            mid = (lo + hi) / 2.0
            o = orness_from_r(mid)
            if (alpha > 0.5 and o < alpha) or (alpha < 0.5 and o > alpha):
                lo = mid
            else:
                hi = mid
            if abs(o - alpha) < tol:
                break
        return weights_from_r((lo + hi) / 2.0)

    # ---------- Models 2–4: Arithmetic progression ----------
    def arith_progression():
        def calc_weights(k):
            d = 2.0 / (k * (k + 1.0))
            w = np.zeros(m)
            j = np.arange(1, k + 1)
            w[:k] = (k - j + 1.0) * d
            return w

        best_k = None
        best_diff = float('inf')
        best_w = None
        for k in range(1, m + 1):
            w = calc_weights(k)
            o = calc_orness(w)
            diff = abs(o - alpha)
            if diff < best_diff or (abs(diff - best_diff) < 1e-12 and o >= alpha):
                best_diff = diff
                best_k = k
                best_w = w
        return best_w

    if model == 1:
        return max_entropy().tolist()
    elif model == 2:
        return arith_progression().tolist()
    else:
        raise ValueError("Model must be 1, 2, 3, or 4.")


owa_weights(2, cutoff, orness)